In [ ]:
import xarray as xr
import hvplot.xarray
import matplotlib.pylab as plt

import cmcrameri.cm  # noqa: F401  pylint: disable=unused-import
import cmglaciology.cm  # noqa: F401  pylint: disable=unused-import


In [ ]:
ds = []
for m in ["mean", "max"]:
    _ds = xr.open_dataset(f"/Users/andy/base/pism-terra/2026_10_plume/output/processed_scalar/{m}_GIS_JIB_g900m_id_OCX_ocx_free_1985-01-01_2005-01-01.nc")
    _ds = _ds.expand_dims({"exp_id": [m]})
    ds.append(_ds)
ds = xr.concat(ds, dim="exp_id").set_index({"glacier_id":"glacier_id_name"}).sel({"glacier_id": "JIB_SHELF"})
ds.glacier_id

In [ ]:
ds

In [ ]:
ds.eigen1.plot(hue="exp_id")

In [ ]:
ds.plume_fresh_water_melt_rate.plot(hue="exp_id")

In [ ]:
ds.velsurf_mag.plot(hue="exp_id")

In [ ]:
spatial = xr.open_dataset("/Users/andy/base/pism-terra/2026_10_plume/output/processed_spatial/spatial_GIS_JIB_g900m_id_OCX_ocx_free_1985-01-01_2005-01-01.nc")
spatial = spatial.sel({"time": slice("1992", "2005")})

In [ ]:
import numpy as np
import panel as pn
import cmglaciology.cm
from matplotlib.colors import LightSource

def hillshade(surface, azdeg=315, altdeg=45, vert_exag=2):
    """Hillshade of every time step of a (time, y, x) surface, 0 (shadow) to 1 (lit)."""
    dx = abs(float(surface.x[1] - surface.x[0]))
    flip = bool(surface.y[0] < surface.y[-1])  # LightSource wants the northern row first
    light = LightSource(azdeg=azdeg, altdeg=altdeg)

    def one(z):
        z = np.where(np.isfinite(z), z, np.nanmin(z))
        z = z[::-1] if flip else z
        shade = light.hillshade(z, vert_exag=vert_exag, dx=dx, dy=dx)
        return shade[::-1] if flip else shade

    shade = xr.apply_ufunc(one, surface, input_core_dims=[["y", "x"]],
                           output_core_dims=[["y", "x"]], vectorize=True)
    return shade.where(surface.notnull()).rename("hillshade").transpose(*surface.dims)

hs = hillshade(spatial.usurf)

background = hs.hvplot.image(x="x", y="y", groupby="time", cmap="gray", clim=(0, 1),
                             colorbar=False, aspect="equal", dynamic=False)
speed = spatial.velsurf_mag.hvplot.image(x="x", y="y", groupby="time", cmap=cmglaciology.cm.speed,
                        clim=(0, 7500), aspect="equal", dynamic=False, alpha=0.8)

pn.panel(background * speed, widget_type="scrubber", widget_location="bottom").embed(
    max_states=240, max_opts=240)

In [ ]:
eigen = spatial.eigen1.hvplot.image(x="x", y="y", groupby="time", cmap=cmcrameri.cm.broc,
                        clim=(-2.5e-8, 2.5e-8), aspect="equal", dynamic=False, alpha=0.8)

pn.panel(background * eigen, widget_type="scrubber", widget_location="bottom").embed(
    max_states=240, max_opts=240)

In [ ]:
import holoviews as hv
import numpy as np
import panel as pn
import cmglaciology.cm
from matplotlib.colors import Normalize

def blend_multiply(rgb, intensity):
    """From glacier_flow_tools.utils: multiply the colors into the hillshade, weighted by their alpha."""
    alpha = rgb[..., -1, np.newaxis]
    img_scaled = np.clip(rgb[..., :3] * intensity, 0.0, 1.0)
    return img_scaled * alpha + intensity * (1.0 - alpha)

cmap = cmglaciology.cm.speed.with_extremes(bad=(0, 0, 0, 0))  # no data: hillshade only
clim = (0, 10000)
norm = Normalize(*clim)

da = spatial.velsurf_mag

frames = {}
for t in da.time.values:
    shade = hs.sel(time=t).values
    rgba = cmap(norm(np.ma.masked_invalid(da.sel(time=t).values)))
    rgb = blend_multiply(rgba, np.nan_to_num(shade, nan=1.0)[..., None])
    inside = np.isfinite(shade)[..., None].astype(float)  # transparent outside the basin
    image = np.concatenate([rgb, inside], axis=-1)
    frames[t] = hv.RGB((da.x.values, da.y.values, *np.moveaxis(image, -1, 0)), kdims=["x", "y"])

blended = hv.HoloMap(frames, kdims="time").opts(aspect="equal", frame_width=500)
# An RGB image has no colorbar; an invisible image on top carries one.
colorbar = da.isel(time=0).hvplot.image(x="x", y="y", cmap=cmap, clim=clim, alpha=0, colorbar=True,
                                        aspect="equal", frame_width=500, clabel="velsurf_mag (m/yr)")

pn.panel(blended * colorbar, widget_type="scrubber", widget_location="bottom").embed(
    max_states=240, max_opts=240)

In [ ]:
cmap = cmglaciology.cm.speed.with_extremes(bad=(0, 0, 0, 0))  # no data: hillshade only
clim = (-2.5e-8, 2.5e-8)
norm = Normalize(*clim)

da = spatial.velsurf_mag

frames = {}
for t in da.time.values:
    shade = hs.sel(time=t).values
    rgba = cmap(norm(np.ma.masked_invalid(da.sel(time=t).values)))
    rgb = blend_multiply(rgba, np.nan_to_num(shade, nan=1.0)[..., None])
    inside = np.isfinite(shade)[..., None].astype(float)  # transparent outside the basin
    image = np.concatenate([rgb, inside], axis=-1)
    frames[t] = hv.RGB((da.x.values, da.y.values, *np.moveaxis(image, -1, 0)), kdims=["x", "y"])

blended = hv.HoloMap(frames, kdims="time").opts(aspect="equal", frame_width=500)
# An RGB image has no colorbar; an invisible image on top carries one.
colorbar = da.isel(time=0).hvplot.image(x="x", y="y", cmap=cmap, clim=clim, alpha=0, colorbar=True,
                                        aspect="equal", frame_width=500, clabel="velsurf_mag (m/yr)")

pn.panel(blended * colorbar, widget_type="scrubber", widget_location="bottom").embed(
    max_states=240, max_opts=240)

In [ ]:
ds.data_vars

In [ ]:
ds_mean = ds.sel(exp_id="mean").sel(time=slice("1992", "2002"))

In [ ]:
ds_mean.ice_mass_transport_across_grounding_line.plot()

In [ ]:
import matplotlib.pylab as plt

In [ ]:
fig, axs = plt.subplots(3, 1, sharex=True, figsize=(12, 8))
ds_mean.velsurf_mag.plot(ax=axs[0])
ds_mean.eigen1.plot(ax=axs[1])
ds_mean.ice_mass_transport_across_grounding_line.plot(ax=axs[2])